##  Uninstall the conflicting packages to clear the slate and reinstall the correct versions

In [ ]:

!pip uninstall -y numpy matplotlib ultralytics opencv-python-headless

!pip install "numpy==1.26.4"

!pip install "matplotlib" "ultralytics" "opencv-python-headless"

## Preprocessed the dataset
>Create the dataset from coco dataset buy taking just the amount we need to finetune the model and the classes that are important for the visually impaired people

In [ ]:
import os
import shutil
import random
from pycocotools.coco import COCO
from tqdm import tqdm

# Paths
coco_annotation_path = '/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_train2017.json'
coco_images_path = '/kaggle/input/coco-2017-dataset/coco2017/train2017'
output_dir = '/kaggle/working/custom_rtdetr_dataset'
images_per_class = 500

target_classes = [
    "person", "handbag", "backpack", "suitcase",
    "bicycle", "car", "motorcycle", "bus", "truck", "train", "airplane",
    "dog", "cat", "bird", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe",
    "chair", "couch", "bed", "toilet", "dining table",
    "tv", "laptop", "mouse", "keyboard", "cell phone",
    "bottle", "cup", "fork", "knife", "spoon", "bowl",
    "skis", "snowboard", "surfboard", "tennis racket", "baseball bat", "frisbee", "kite", "skateboard",
    "clock", "book", "umbrella"
]

# Create the set of classes we need to extract
needed_coco_classes = set(target_classes)

def create_dataset():
    # Initialize COCO api
    coco = COCO(coco_annotation_path)
    
    # Get ID for each category name
    # We explicitly convert the set to a list to avoid ordering issues
    cat_ids = coco.getCatIds(catNms=list(needed_coco_classes))
    cats = coco.loadCats(cat_ids)
    
    # Create a lookup: COCO_ID -> New_Class_ID (0, 1, 2...)
    cat_id_to_new_id = {cat['id']: i for i, cat in enumerate(cats)}
    new_id_to_name = {i: cat['name'] for i, cat in enumerate(cats)}
    
    # Prepare directories
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir) # Clean up previous runs
    os.makedirs(f"{output_dir}/images/train", exist_ok=True)
    os.makedirs(f"{output_dir}/labels/train", exist_ok=True)

    # Track how many images we have for each class
    class_counts = {cat['name']: 0 for cat in cats}
    processed_image_ids = set()

    print("Selecting images...")
    
    # Iterate through each category
    for cat in tqdm(cats):
        cat_name = cat['name']
        cat_id = cat['id']
        
        # Get all image IDs containing this category
        img_ids = coco.getImgIds(catIds=[cat_id])
        random.shuffle(img_ids) 
        
        for img_id in img_ids:
            # Stop if we have enough for this class
            if class_counts[cat_name] >= images_per_class:
                break
                
            # Skip if we already processed this image
            if img_id in processed_image_ids:
                continue

            # Load Image Info
            img_info = coco.loadImgs(img_id)[0]
            file_name = img_info['file_name']
            
            # COPY IMAGE
            src_img = os.path.join(coco_images_path, file_name)
            if not os.path.exists(src_img):
                continue
                
            dst_img = os.path.join(output_dir, "images/train", file_name)
            shutil.copy(src_img, dst_img)
            
            # GENERATE LABEL (YOLO FORMAT)
            ann_ids = coco.getAnnIds(imgIds=img_id, catIds=cat_ids)
            anns = coco.loadAnns(ann_ids)
            
            label_file = file_name.replace('.jpg', '.txt')
            label_path = os.path.join(output_dir, "labels/train", label_file)
            
            has_relevant_obj = False
            with open(label_path, 'w') as f:
                for ann in anns:
                    # BBox calculations
                    x_min, y_min, w, h = ann['bbox']
                    
                    img_w, img_h = img_info['width'], img_info['height']
                    x_center = (x_min + w / 2) / img_w
                    y_center = (y_min + h / 2) / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h
                    
                    # Get the new class index (0-N)
                    class_idx = cat_id_to_new_id[ann['category_id']]
                    
                    f.write(f"{class_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                    
                    # Update counts
                    current_cat_name = new_id_to_name[class_idx]
                    class_counts[current_cat_name] += 1
                    has_relevant_obj = True

            if has_relevant_obj:
                processed_image_ids.add(img_id)

    # CREATE DATA.YAML for RT-DETR
    yaml_content = f"""
            path: {output_dir} 
            train: images/train
            val: images/train 
            
            nc: {len(cats)}
            names:
            """
    for i in range(len(cats)):
        yaml_content += f"  {i}: {new_id_to_name[i]}\n"

    with open(f"{output_dir}/data.yaml", "w") as f:
        f.write(yaml_content)

if __name__ == "__main__":
    create_dataset()

### Finetune the RTDETR model

In [ ]:
from ultralytics import RTDETR

# Load fresh model
model = RTDETR('rtdetr-l.pt')

# Train
results = model.train(
    data='/kaggle/working/custom_rtdetr_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    project='/kaggle/working/runs/train',
    name='rtdetr_fresh_start',
    device=0,
    amp=False 
)

## Train for more iterations

In [ ]:
from ultralytics import RTDETR

# LAST saved checkpoint
checkpoint_path = '/kaggle/working/runs/train/rtdetr_custom_finetune7/weights/last.pt'

try:
    # Try to load the checkpoint (if it exists)
    print(f"Resuming from checkpoint: {checkpoint_path}")
    model = RTDETR(checkpoint_path)
    resume_training = True
except Exception as e:
    # If file doesn't exist (first run), load the base model
    print(f"The file doesn't exist in that path ")

# Train with resume=True
results = model.train(
    data='/kaggle/working/custom_rtdetr_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    project='/kaggle/working/runs/train',
    name='rtdetr_custom_finetune',
    device=0,
    amp=False,
    resume=True
)

## Results of all epocs

In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/working/runs/train/rtdetr_fresh_start/results.csv")
print(df)

    epoch       time  train/giou_loss  train/cls_loss  train/l1_loss  \
0       1    363.399          0.78648         2.07374        0.35845   
1       2    723.046          0.58992         0.69857        0.25496   
2       3   1081.830          0.57828         0.70613        0.24728   
3       4   1439.960          0.55394         0.69963        0.24200   
4       5   1797.940          0.55228         0.69402        0.23655   
5       6   2155.730          0.53579         0.68763        0.22868   
6       7   2513.460          0.52594         0.67506        0.22414   
7       8   2871.080          0.51813         0.65910        0.21825   
8       9   3228.920          0.51823         0.64153        0.21186   
9      10   3587.530          0.51392         0.63405        0.21183   
10     11   3945.590          0.50087         0.62247        0.20574   
11     12   4303.280          0.49343         0.61504        0.20319   
12     13   4661.140          0.49049         0.60867        0.1

## Test the model for inference

In [13]:
# 1. Install the downloader
!pip install yt-dlp -q

# 2. Download a specific video (NYC Walking Tour)
# We limit it to the first 30 seconds to save processing time
print("Downloading video...")
!yt-dlp "https://www.youtube.com/watch?v=F8MN0o6RS9o" \
    -o "street_test.mp4" \
    --download-sections "*00:00-00:30" \
    --format "best[ext=mp4]" --force-overwrite

print("Video downloaded as: street_test.mp4")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.4 MB/s eta 0:00:00:00:01
[youtube] Extracting URL: https://www.youtube.com/watch?v=F8MN0o6RS9o
[youtube] F8MN0o6RS9o: Downloading webpage
[youtube] F8MN0o6RS9o: Downloading android sdkless player API JSON
[youtube] F8MN0o6RS9o: Downloading web safari player API JSON
[youtube] F8MN0o6RS9o: Downloading m3u8 information
[info] F8MN0o6RS9o: Downloading 1 format(s): 301
[info] F8MN0o6RS9o: Downloading 1 time ranges: 0.0-30.0
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: street_test.mp4
[hls @ 0x5aefc2abc5c0] Skip ('#EXT-X-VERSION:3')
[hls @ 0x5aefc2abc5c0] Opening 'https://rr5---sn-qxoednel.googlevideo.com/videoplayback/id/17c30dd28e914bda/itag/301/source/youtube/expire/1764809729/ei/oYcwaaD1F8SYsfIPveKsqQE/ip/35.185.202.87/requiressl/yes/ratebypass/yes/pfa/1/sgoap/clen%3D72834623%3Bdur%3D4500.398%3Bgir%3Dyes%3Bitag%

In [15]:
from ultralytics import RTDETR
import os

# 1. Load your BEST model
# Update this path if you changed the project name!
model_path = '/kaggle/working/runs/train/rtdetr_fresh_start/weights/best.pt'

if not os.path.exists(model_path):
    print("Warning: best.pt not found yet. Using last.pt...")

model = RTDETR(model_path)

# 2. Run Inference
results = model.predict(
    source='street_test.mp4',
    save=True,
    conf=0.4,
    project='/kaggle/working/',
    name='video_test',
    exist_ok=True
)


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1799) /kaggle/working/street_test.mp4: 640x640 (no detections), 45.4ms
video 1/1 (frame 2/1799) /kaggle/working/street_test.mp4: 640x640 (no detections), 45.1ms
video 1/1 (frame 3/1799) /kaggle/working/street_test.mp4: 640x640 (no detections), 47.3ms
video 1/1 (frame 4/1799) /kaggle/working/street_test.mp4: 640x640 1 car, 44.9ms
video 1/1 (frame 5/1799) /kaggle/working/street_test.mp4: 640x640 2 cars, 45.6ms
video 1/1 (frame 6/1799) 